# Fairness-Aware Property Assessment Modeling in Cook County

## 1. Project Motivation

Property valuation models affect more than predictive accuracy. In a property-tax system, systematic overvaluation or undervaluation can translate into unequal tax burdens across homeowners and communities.

A model may achieve strong aggregate predictive performance while producing systematically different errors for different property-value or demographic groups. Therefore, evaluating a property assessment model requires examining both:

1. **Predictive accuracy** — how closely predicted property values match observed sale prices.
2. **Assessment fairness** — whether the magnitude and direction of prediction errors differ systematically across groups.

This project develops and evaluates property valuation models using residential property sales from Cook County, Illinois. In addition to property characteristics, Census tract-level demographic information will be incorporated to examine whether model errors differ across socioeconomic and demographic contexts.

The project will ultimately compare:

- a conventional linear-regression baseline,
- a stronger predictive benchmark,
- and a fairness-aware model that explicitly penalizes undesirable directional assessment errors.

The central research question is:

> **Can a property valuation model maintain competitive predictive accuracy while reducing systematic assessment disparities across property-value and demographic groups?**

## 2. What Does "Fair" Mean in This Project?

Fairness in property assessment is not equivalent to requiring every individual prediction to be correct.

Instead, we are interested in whether prediction errors exhibit systematic patterns across groups.

For a property with observed sale price $y_i$ and predicted value $\hat{y}_i$:

- $ \hat{y}_i > y_i\ $ represents **overassessment**
- $ \hat{y}_i < y_i\ $ represents **underassessment**

A model may have reasonable overall RMSE while still disproportionately overassessing one segment of the housing market.

We will therefore evaluate models along several dimensions:

### Predictive Performance
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- Mean Absolute Percentage Error (MAPE)

### Directional Assessment Behavior
- Mean residual
- Overassessment rate
- Underassessment rate
- Assessment ratio

$$
\text{Assessment Ratio}_i = \frac{\hat{y}_i}{y_i}
$$

An assessment ratio greater than 1 indicates overassessment, while a ratio below 1 indicates underassessment.

### Group-Level Fairness

These quantities will later be compared across:

1. Property-value groups
2. Census tract income groups
3. Census tract demographic composition
4. Geographic areas

The goal is not to assume that property value itself represents race or income. Instead, demographic conclusions will only be made after explicitly joining external Census/ACS demographic data.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os
import requests

In [17]:
with zipfile.ZipFile('cook_county_data.zip', 'r') as z:
    with z.open("cook_county_train.csv") as f:
        data = pd.read_csv(f)

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (204792, 63)


,Unnamed: 0,PIN,Property Class,Neighborhood Code,Land Square Feet,Town Code,Apartments,Wall Material,Roof Material,Basement,...,Sale Month of Year,Sale Half of Year,Most Recent Sale,Age Decade,Pure Market Filter,Garage Indicator,Neigborhood Code (mapping),Town and Neighborhood,Description,Lot Size
0,0,17294100610000,203,50,2500.0,76,0.0,2.0,1.0,1.0,...,9,2,1.0,13.2,0,0.0,50,7650,"This property, sold on 09/14/2015, is a one-st...",2500.0
1,1,13272240180000,202,120,3780.0,71,0.0,2.0,1.0,1.0,...,5,1,1.0,9.6,1,1.0,120,71120,"This property, sold on 05/23/2018, is a one-st...",3780.0
2,2,25221150230000,202,210,4375.0,70,0.0,2.0,1.0,2.0,...,2,1,0.0,11.2,1,1.0,210,70210,"This property, sold on 02/18/2016, is a one-st...",4375.0
3,3,10251130030000,203,220,4375.0,17,0.0,3.0,1.0,1.0,...,7,2,1.0,6.3,1,1.0,220,17220,"This property, sold on 07/23/2013, is a one-st...",4375.0
4,4,31361040550000,202,120,8400.0,32,0.0,3.0,1.0,2.0,...,6,1,0.0,6.3,1,1.0,120,32120,"This property, sold on 06/10/2016, is a one-st...",8400.0


In [18]:
print(f"Number of observations: {data.shape[0]:,}")
print(f"Number of columns: {data.shape[1]}")

data.info()

Number of observations: 204,792
Number of columns: 63
<class 'pandas.DataFrame'>
RangeIndex: 204792 entries, 0 to 204791
Data columns (total 63 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Unnamed: 0                  204792 non-null  int64  
 1   PIN                         204792 non-null  int64  
 2   Property Class              204792 non-null  int64  
 3   Neighborhood Code           204792 non-null  int64  
 4   Land Square Feet            204792 non-null  float64
 5   Town Code                   204792 non-null  int64  
 6   Apartments                  204792 non-null  float64
 7   Wall Material               204792 non-null  float64
 8   Roof Material               204792 non-null  float64
 9   Basement                    204792 non-null  float64
 10  Basement Finish             204792 non-null  float64
 11  Central Heating             204792 non-null  float64
 12  Other Heating               2

## 4. Core Variables

The raw dataset contains property characteristics, transaction information, geographic identifiers, and assessment-related variables.

Before performing exploratory analysis, we first identify the variables most relevant to the project:

### Outcome
- `Sale Price`

### Property Characteristics
Examples include:
- `Building Square Feet`
- `Land Square Feet`
- `Bathrooms`
- `Age`
- `Property Class`
- construction and condition variables
- garage and improvement characteristics

### Geography
- `Census Tract`
- `Latitude`
- `Longitude`
- `Town Code`
- `Neighborhood Code`

### Time
- `Sale Year`
- `Sale Quarter`
- `Sale Month of Year`

The `Census Tract` variable is especially important because it potentially provides a direct key for joining tract-level demographic information from the American Community Survey.

In [19]:
{
    "shape": data.shape,
    "sale_year_range": (
        data["Sale Year"].min(),
        data["Sale Year"].max()
    ),
    "n_census_tracts": data["Census Tract"].nunique(),
    "missing_census_tract_pct": data["Census Tract"].isna().mean() * 100,
    "missing_latitude_pct": data["Latitude"].isna().mean() * 100,
    "missing_longitude_pct": data["Longitude"].isna().mean() * 100,
}

{'shape': (204792, 63),
 'sale_year_range': (np.int64(2013), np.int64(2019)),
 'n_census_tracts': 1265,
 'missing_census_tract_pct': np.float64(0.0),
 'missing_latitude_pct': np.float64(0.0),
 'missing_longitude_pct': np.float64(0.0)}

In [20]:
data[
    [
        "Census Tract",
        "Latitude",
        "Longitude",
        "Town Code",
        "Neighborhood Code",
        "Sale Year",
        "Sale Price"
    ]
].head(10)

,Census Tract,Latitude,Longitude,Town Code,Neighborhood Code,Sale Year,Sale Price
0,600600.0,41.840803,-87.654264,76,50,2015,1
1,200100.0,41.933016,-87.735966,71,120,2018,285000
2,491400.0,41.686738,-87.616496,70,210,2016,22000
3,810301.0,42.019937,-87.701482,17,220,2013,225000
4,830300.0,41.476430,-87.682236,32,120,2016,22600
5,640300.0,41.784580,-87.769661,72,380,2018,1
6,828202.0,41.557900,-87.547981,37,181,2017,100000
7,801606.0,42.118618,-87.855616,25,52,2016,795000
8,160800.0,41.953313,-87.712016,71,70,2016,675000
9,804105.0,42.091755,-88.101858,29,33,2019,270000


### Examining Data Structure/Quality

The raw Cook County dataset contains **204792 property-sale observations and 63 variables**, covering transactions from **2013 through 2019**.

Geographic coverage is strong for the planned demographic analysis:

- The dataset contains **1,265 unique Census tracts**.
- Census tract, latitude, and longitude information are complete, with **0% missingness**.
- Therefore, tract-level demographic enrichment appears feasible without discarding a meaningful portion of the housing dataset.

The initial inspection also reveals transactions recorded at extremely small sale prices, including \$1 sales. These likely represent non-market or nominal transactions and should be investigated during the data-cleaning stage rather than treated as ordinary residential sales.

## 5. Census Geography Validation

To incorporate demographic information, each property must be linked to a Census tract.

The Census Bureau identifies Census tracts using a six-digit tract code within a state and county. For Cook County:

- Illinois state FIPS code: `17`
- Cook County FIPS code: `031`

The housing dataset stores Census tract identifiers numerically, which may remove leading zeroes. We therefore standardize the tract variable as a six-character string before attempting the demographic join.

The complete tract GEOID can then be represented as:

$$
\text{GEOID} =
\text{State FIPS} +
\text{County FIPS} +
\text{Tract Code}
$$

For Cook County, this gives an 11-digit identifier beginning with `17031`.

In [21]:
# Preserve missing values while creating a standardized six-digit tract code
data["Census Tract Code"] = (
    data["Census Tract"]
    .astype("Int64")
    .astype("string")
    .str.zfill(6)
)

# Construct full 11-digit Census tract GEOID
data["Census Tract GEOID"] = (
    "17031" + data["Census Tract Code"]
)

data[
    [
        "Census Tract",
        "Census Tract Code",
        "Census Tract GEOID"
    ]
].head(10)

,Census Tract,Census Tract Code,Census Tract GEOID
0,600600.0,600600,17031600600
1,200100.0,200100,17031200100
2,491400.0,491400,17031491400
3,810301.0,810301,17031810301
4,830300.0,830300,17031830300
5,640300.0,640300,17031640300
6,828202.0,828202,17031828202
7,801606.0,801606,17031801606
8,160800.0,160800,17031160800
9,804105.0,804105,17031804105


In [22]:
print("Unique raw tracts:", data["Census Tract"].nunique())
print("Unique standardized tracts:", data["Census Tract Code"].nunique())

print("\nTract-code length distribution:")
print(data["Census Tract Code"].str.len().value_counts(dropna=False))

print("\nSample standardized tracts:")
print(
    data[
        ["Census Tract", "Census Tract Code", "Census Tract GEOID"]
    ]
    .drop_duplicates()
    .sort_values("Census Tract Code")
    .head(15)
)

Unique raw tracts: 1265
Unique standardized tracts: 1265

Tract-code length distribution:
Census Tract Code
6    204792
Name: count, dtype: Int64

Sample standardized tracts:
        Census Tract Census Tract Code Census Tract GEOID
6603         10100.0            010100        17031010100
2679         10201.0            010201        17031010201
2644         10202.0            010202        17031010202
9933         10300.0            010300        17031010300
14568        10400.0            010400        17031010400
63862        10501.0            010501        17031010501
40883        10502.0            010502        17031010502
179851       10503.0            010503        17031010503
2877         10600.0            010600        17031010600
816          10701.0            010701        17031010701
10303        10702.0            010702        17031010702
18515        20100.0            020100        17031020100
237          20200.0            020200        17031020200
1696         

### Census Tract Identifier Validation

Standardizing the Census tract variable successfully preserved all **1,265 unique tract identifiers** while converting each non-missing value to the six-digit format used by the Census Bureau.

Of the 204792 property-sale observations, **all have a valid six-digit tract code**, with only one observation missing geographic information. The resulting 11-digit GEOID combines the Illinois state FIPS (`17`), Cook County FIPS (`031`), and six-digit tract code.

Because no tract identifiers were lost or duplicated during standardization, the resulting GEOID is a suitable candidate key for joining external tract-level demographic data.

## 6. Validate Property Tracts Against Official Census Geography

Before attaching demographic variables, the locally constructed Census GEOIDs should be compared against an official list of Census tracts.

This validation serves two purposes:

1. It verifies that the housing dataset's tract coding is compatible with Census geography.
2. It allows us to measure the expected demographic-data match rate before performing the actual merge.

We use the 2019 ACS 5-year geography because the housing transactions span 2013–2019 and the 2019 release provides a reasonable demographic snapshot near the end of the study period.

In [23]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

print("Key loaded:", CENSUS_API_KEY is not None)

Key loaded: True


In [24]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")

url = "https://api.census.gov/data/2019/acs/acs5"

params = {
    "get": "NAME,B01001_001E",
    "for": "tract:*",
    "in": "state:17 county:031",
    "key": CENSUS_API_KEY
}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Final URL:", response.url)

response.raise_for_status()

Status code: 200
Content type: application/json;charset=utf-8
Final URL: https://api.census.gov/data/2019/acs/acs5?get=NAME%2CB01001_001E&for=tract%3A%2A&in=state%3A17+county%3A031&key=d8c5bdf38a865a01095ff5760e83486f9046c9d5


In [25]:
acs_raw = response.json()

acs_tracts = pd.DataFrame(
    acs_raw[1:],
    columns=acs_raw[0]
)

acs_tracts["Census Tract GEOID"] = (
    acs_tracts["state"]
    + acs_tracts["county"]
    + acs_tracts["tract"]
)

print("Number of official Cook County ACS tracts:", len(acs_tracts))
acs_tracts.head()

Number of official Cook County ACS tracts: 1319


,NAME,B01001_001E,state,county,tract,Census Tract GEOID
0,"Census Tract 6302, Cook County, Illinois",1825,17,031,630200,17031630200
1,"Census Tract 5807, Cook County, Illinois",5908,17,031,580700,17031580700
2,"Census Tract 5906, Cook County, Illinois",3419,17,031,590600,17031590600
3,"Census Tract 6007, Cook County, Illinois",2835,17,031,600700,17031600700
4,"Census Tract 6119, Cook County, Illinois",1639,17,031,611900,17031611900


In [26]:
property_geoids = set(
    data["Census Tract GEOID"].dropna()
)

acs_geoids = set(
    acs_tracts["Census Tract GEOID"]
)

matched_geoids = property_geoids & acs_geoids
unmatched_geoids = property_geoids - acs_geoids

tract_match_rate = (
    len(matched_geoids) / len(property_geoids) * 100
)

property_match_rate = (
    data["Census Tract GEOID"].isin(acs_geoids).mean() * 100
)

print("Unique property tracts:", len(property_geoids))
print("Matched unique tracts:", len(matched_geoids))
print("Unmatched unique tracts:", len(unmatched_geoids))
print(f"Tract-level match rate: {tract_match_rate:.2f}%")
print(f"Property-level match rate: {property_match_rate:.2f}%")

print("\nSample unmatched tract GEOIDs:")
print(sorted(unmatched_geoids)[:20])

Unique property tracts: 1265
Matched unique tracts: 1265
Unmatched unique tracts: 0
Tract-level match rate: 100.00%
Property-level match rate: 100.00%

Sample unmatched tract GEOIDs:
[]


## 7. Constructing Census-Based Fairness Variables

The successful GEOID validation shows that all census tracts represented in the
Cook County property dataset can be linked to the 2019 ACS 5-Year Estimates.

The next step is to construct tract-level demographic and socioeconomic
variables that can later be used to evaluate whether prediction errors are
systematically distributed across different communities.

These variables are initially treated as **fairness evaluation attributes**
rather than housing-price predictors. This distinction allows the predictive
model to rely primarily on property characteristics while demographic context
is used to examine whether model errors disproportionately affect particular
neighborhoods.

The first set of ACS measures captures:

- population,
- median household income,
- poverty,
- racial and ethnic composition.

In [27]:
acs_variables = {
    # Population
    "B01001_001E": "total_population",

    # Median household income
    "B19013_001E": "median_household_income",

    # Poverty
    "B17001_001E": "poverty_universe",
    "B17001_002E": "below_poverty",

    # Race / ethnicity
    "B03002_001E": "race_ethnicity_total",
    "B03002_003E": "white_non_hispanic",
    "B03002_004E": "black_non_hispanic",
    "B03002_006E": "asian_non_hispanic",
    "B03002_012E": "hispanic"
}

params = {
    "get": "NAME," + ",".join(acs_variables.keys()),
    "for": "tract:*",
    "in": "state:17 county:031",
    "key": CENSUS_API_KEY
}

response = requests.get(
    "https://api.census.gov/data/2019/acs/acs5",
    params=params,
    timeout=30
)

print("Status code:", response.status_code)
response.raise_for_status()

Status code: 200


In [28]:
acs_raw = response.json()

acs_fairness = pd.DataFrame(
    acs_raw[1:],
    columns=acs_raw[0]
)

acs_fairness = acs_fairness.rename(columns=acs_variables)

acs_fairness["Census Tract GEOID"] = (
    acs_fairness["state"]
    + acs_fairness["county"]
    + acs_fairness["tract"]
)

acs_fairness.head()

,NAME,total_population,median_household_income,poverty_universe,below_poverty,race_ethnicity_total,white_non_hispanic,black_non_hispanic,asian_non_hispanic,hispanic,state,county,tract,Census Tract GEOID
0,"Census Tract 6302, Cook County, Illinois",1825,37422,1825,436,1825,125,0,78,1622,17,031,630200,17031630200
1,"Census Tract 5807, Cook County, Illinois",5908,47000,5908,1216,5908,423,161,522,4742,17,031,580700,17031580700
2,"Census Tract 5906, Cook County, Illinois",3419,46033,3416,404,3419,757,9,431,2119,17,031,590600,17031590600
3,"Census Tract 6007, Cook County, Illinois",2835,45294,2835,432,2835,1001,82,857,850,17,031,600700,17031600700
4,"Census Tract 6119, Cook County, Illinois",1639,24507,1639,766,1639,26,1175,0,438,17,031,611900,17031611900


In [29]:
numeric_cols = list(acs_variables.values())

for col in numeric_cols:
    acs_fairness[col] = pd.to_numeric(
        acs_fairness[col],
        errors="coerce"
    )

In [34]:
acs_fairness["poverty_rate"] = (
    acs_fairness["below_poverty"]
    / acs_fairness["poverty_universe"]
)

acs_fairness["pct_white_non_hispanic"] = (
    acs_fairness["white_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_black_non_hispanic"] = (
    acs_fairness["black_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_asian_non_hispanic"] = (
    acs_fairness["asian_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_hispanic"] = (
    acs_fairness["hispanic"]
    / acs_fairness["race_ethnicity_total"]
)
acs_fairness.head()

,NAME,total_population,median_household_income,poverty_universe,below_poverty,race_ethnicity_total,white_non_hispanic,black_non_hispanic,asian_non_hispanic,hispanic,state,county,tract,Census Tract GEOID,poverty_rate,pct_white_non_hispanic,pct_black_non_hispanic,pct_asian_non_hispanic,pct_hispanic
0,"Census Tract 6302, Cook County, Illinois",1825,37422,1825,436,1825,125,0,78,1622,17,031,630200,17031630200,0.238904,0.068493,0.000000,0.042740,0.888767
1,"Census Tract 5807, Cook County, Illinois",5908,47000,5908,1216,5908,423,161,522,4742,17,031,580700,17031580700,0.205823,0.071598,0.027251,0.088355,0.802640
2,"Census Tract 5906, Cook County, Illinois",3419,46033,3416,404,3419,757,9,431,2119,17,031,590600,17031590600,0.118267,0.221410,0.002632,0.126060,0.619772
3,"Census Tract 6007, Cook County, Illinois",2835,45294,2835,432,2835,1001,82,857,850,17,031,600700,17031600700,0.152381,0.353086,0.028924,0.302293,0.299824
4,"Census Tract 6119, Cook County, Illinois",1639,24507,1639,766,1639,26,1175,0,438,17,031,611900,17031611900,0.467358,0.015863,0.716901,0.000000,0.267236


In [31]:
fairness_cols = [
    "total_population",
    "median_household_income",
    "poverty_rate",
    "pct_white_non_hispanic",
    "pct_black_non_hispanic",
    "pct_asian_non_hispanic",
    "pct_hispanic"
]

acs_fairness[fairness_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
total_population,1319.0,3.941073e+03,1.878442e+03,0.0,2506.500000,3818.000000,5159.500000,20087.000000
median_household_income,1319.0,-2.459312e+06,4.098792e+07,-666666666.0,42301.000000,60000.000000,86348.000000,250001.000000
poverty_rate,1315.0,1.624773e-01,1.249615e-01,0.0,0.066822,0.127010,0.227102,0.749475
pct_white_non_hispanic,1315.0,3.911069e-01,3.085193e-01,0.0,0.062832,0.378989,0.682237,0.967855
pct_black_non_hispanic,1315.0,2.845208e-01,3.625682e-01,0.0,0.018412,0.058403,0.610188,1.000000
pct_asian_non_hispanic,1315.0,6.462926e-02,9.493854e-02,0.0,0.002706,0.027646,0.085808,0.863344
pct_hispanic,1315.0,2.386868e-01,2.622091e-01,0.0,0.050865,0.124526,0.332129,0.991284


In [32]:
acs_fairness[fairness_cols].isna().sum()

total_population           0
median_household_income    0
poverty_rate               4
pct_white_non_hispanic     4
pct_black_non_hispanic     4
pct_asian_non_hispanic     4
pct_hispanic               4
dtype: int64

In [33]:
for col in fairness_cols:
    print(
        f"{col}: "
        f"min={acs_fairness[col].min():.3f}, "
        f"median={acs_fairness[col].median():.3f}, "
        f"max={acs_fairness[col].max():.3f}"
    )

total_population: min=0.000, median=3818.000, max=20087.000
median_household_income: min=-666666666.000, median=60000.000, max=250001.000
poverty_rate: min=0.000, median=0.127, max=0.749
pct_white_non_hispanic: min=0.000, median=0.379, max=0.968
pct_black_non_hispanic: min=0.000, median=0.058, max=1.000
pct_asian_non_hispanic: min=0.000, median=0.028, max=0.863
pct_hispanic: min=0.000, median=0.125, max=0.991


## 8. Cleaning ACS Special Values

Inspection of the retrieved ACS variables identified a special negative value in
`median_household_income`. This value does not represent an actual household
income; it is an ACS sentinel value indicating that the estimate is unavailable
or not applicable.

Before merging the demographic data with the property records, invalid ACS
sentinel values are converted to missing values. We also inspect tracts with
undefined demographic rates to determine whether they result from zero
population denominators.

In [35]:
# ACS estimates should not contain negative values for any of the
# demographic measures used in this project.
acs_numeric_cols = [
    "total_population",
    "median_household_income",
    "poverty_universe",
    "below_poverty",
    "race_ethnicity_total",
    "white_non_hispanic",
    "black_non_hispanic",
    "asian_non_hispanic",
    "hispanic"
]

for col in acs_numeric_cols:
    acs_fairness.loc[acs_fairness[col] < 0, col] = np.nan

In [37]:
# Because our rates were computed before this cleaning, recalculate them
acs_fairness["poverty_rate"] = (
    acs_fairness["below_poverty"]
    / acs_fairness["poverty_universe"]
)

acs_fairness["pct_white_non_hispanic"] = (
    acs_fairness["white_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_black_non_hispanic"] = (
    acs_fairness["black_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_asian_non_hispanic"] = (
    acs_fairness["asian_non_hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

acs_fairness["pct_hispanic"] = (
    acs_fairness["hispanic"]
    / acs_fairness["race_ethnicity_total"]
)

In [38]:
print("Missing values after ACS cleaning:")
print(acs_fairness[fairness_cols].isna().sum())

print("\nTracts with undefined demographic rates:")
acs_fairness.loc[
    acs_fairness[
        [
            "poverty_rate",
            "pct_white_non_hispanic",
            "pct_black_non_hispanic",
            "pct_asian_non_hispanic",
            "pct_hispanic"
        ]
    ].isna().any(axis=1),
    [
        "NAME",
        "Census Tract GEOID",
        "total_population",
        "poverty_universe",
        "race_ethnicity_total",
        "median_household_income"
    ]
]

Missing values after ACS cleaning:
total_population           0
median_household_income    5
poverty_rate               4
pct_white_non_hispanic     4
pct_black_non_hispanic     4
pct_asian_non_hispanic     4
pct_hispanic               4
dtype: int64

Tracts with undefined demographic rates:


,NAME,Census Tract GEOID,total_population,poverty_universe,race_ethnicity_total,median_household_income
484,"Census Tract 9801, Cook County, Illinois",17031980100,0.0,0.0,0.0,NaN
485,"Census Tract 9800, Cook County, Illinois",17031980000,0.0,0.0,0.0,NaN
486,"Census Tract 9900, Cook County, Illinois",17031990000,0.0,0.0,0.0,NaN
845,"Census Tract 3817, Cook County, Illinois",17031381700,0.0,0.0,0.0,NaN


## 9. Merge ACS Demographic Context with Property Records

After validating Census tract identifiers and cleaning ACS special values,
the tract-level demographic variables are merged onto the Cook County property
dataset.

The merge uses `Census Tract GEOID` as the geographic key. Because each ACS
row represents one Census tract while the property dataset may contain many
properties within the same tract, this is a many-to-one merge.

A left join is used so that every property record is retained. Merge integrity
is then evaluated by checking:

- row counts before and after the merge,
- uniqueness of ACS tract identifiers,
- Census variable coverage,
- and the merge indicator.